# Sección 1.A : Visualización de Datos, Exploración y entendimiento del DataSet

In [ ]:
# Importar librerías necesarias para análisis de datos
import pandas as pd
import matplotlib.pyplot as plt
import os

# Verificar el directorio actual de trabajo
print("Directorio actual de trabajo:", os.getcwd())

# Intentar cargar el archivo CSV con diferentes rutas posibles
try:
    # Primera opción: ruta directa desde la raíz del proyecto
    path = 'data/01_raw/default_of_credit_card_clients.csv'
    data = pd.read_csv(path, header=1)
    print("¡Archivo cargado con éxito desde la ruta directa!")
except FileNotFoundError:
    try:
        # Segunda opción: subir un nivel si el notebook está en subcarpeta
        path = '../data/01_raw/default_of_credit_card_clients.csv'
        data = pd.read_csv(path, header=1)
        print("¡Archivo cargado con éxito subiendo un nivel (../)!")
    except FileNotFoundError:
        print("Error: Aún no se encuentra el archivo. Revisa la estructura de carpetas.")

# Mostrar información básica del dataset si se cargó correctamente
if 'data' in locals():
    print("Columnas del dataset:", data.columns.tolist())
    print(f"Número total de columnas: {len(data.columns)}")
    print("Información general del dataset:")
    data.info()
    print("Estadísticas descriptivas:")
    data.describe()

# Verificar valores nulos en cada columna
print("Valores nulos por columna:")
print(data.isnull().sum())

# Mostrar métricas estadísticas detalladas
print("\nMétricas estadísticas completas:")
print(data.describe())

# Crear histogramas para visualizar la distribución de variables numéricas
print("Creando histogramas para todas las columnas numéricas...")
data.hist(figsize=(15, 10), bins=30)
plt.tight_layout()
plt.show()

## Detalles de cada **Feature**

In [ ]:
# Importar numpy para operaciones numéricas
import numpy as np

# Análisis detallado de cada variable (columna) del dataset

# Identificar columnas que contienen números
numerical_cols = data.select_dtypes(include=[np.number]).columns

# Analizar cada columna numérica
for col in numerical_cols:
    print(f"\n--- Análisis de la columna {col} ---")
    print("Estadísticas básicas:")
    print(data[col].describe())
    print(f"Cantidad de valores faltantes: {data[col].isnull().sum()}")

    # Crear gráficos para visualizar la distribución
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Histograma: muestra la frecuencia de valores
    axes[0].hist(data[col], bins=30, edgecolor='black')
    axes[0].set_title(f'Distribución de {col}')
    axes[0].set_xlabel(col)
    axes[0].set_ylabel('Frecuencia')

    # Boxplot: identifica valores atípicos y rangos
    axes[1].boxplot(data[col])
    axes[1].set_title(f'Valores de {col}')
    axes[1].set_ylabel(col)

    plt.tight_layout()
    plt.show()

# Verificar si hay columnas categóricas para analizar
categorical_cols = data.select_dtypes(include=['object', 'category']).columns
if len(categorical_cols) > 0:
    # Analizar cada columna categórica
    for col in categorical_cols:
        print(f"\n--- Análisis de {col} (variable categórica) ---")
        print("Conteo de cada categoría:")
        print(data[col].value_counts())
        # Gráfico de barras para ver distribución
        data[col].value_counts().plot(kind='bar', figsize=(8, 4))
        plt.title(f'Distribución de categorías en {col}')
        plt.show()
else:
    print("\nNo se encontraron columnas categóricas en el dataset.")

# Sección 1.B : Análisis del primer vistazo al dataset.

## Detalles observados:
Existe una columna llamada 'ID', para el entrenamiento del modelo no será relevante por lo tanto está columna/feature será considerada para su eliminación.

Podemos observar que existen 3 features de tipo categórico-nominal específicamente: "SEX", "EDUCATION", "MARRIAGE". Debido a esto tomaremos la decisión de aplicar la estrategia de One-Hot Encoding para mejorar el dataset para luego utilizarlo para entrenar al modelo.

Para el feature "SEX", se proponen los siguientes features: "SEX_MAN", "SEX_WOMAN".
Para el feature "EDUCATION", se proponen los siguientes features: "EDUCATION_GRAD_SCHOOL", "EDUCATION_HIGH_SCHOOL", "EDUCATION_OTHERS", "EDUCATION_UNIVERSITY"
Para el feature "MARRIAGE", se proponen los siguientes featurse: "MARRIAGE_MARRIED", "MARRIAGE_OTHERS", "MARRIAGE_SINGLE"

## Se propone aplicar featuring engineering para aportar más valor al dataset. En este caso lo siguientes:

**A. Ratio de Utilización de Crédito (utilization_ratio)**
Qué es:
La división entre cuánto debe el cliente (BILL_AMT) y cuánto tiene permitido gastar (LIMIT_BAL).

Por qué es vital: Un cliente que debe $9,000 y tiene un límite de $10,000 (90% de uso) es mucho más riesgoso que uno que debe los mismos $9,000 pero tiene un límite de $100,000 (9% de uso). El valor absoluto es el mismo, pero el contexto de riesgo es totalmente distinto.

---

**B. Ratio de Pago (pay_ratio)**
Qué es: Relación entre lo que el cliente pagó este mes y lo que debía el mes anterior.

Por qué es vital: Nos dice si el cliente paga el total de su deuda, solo el mínimo, o nada. Si el ratio es constantemente bajo (cercano a 0), la deuda se está acumulando (interés compuesto), lo que precede casi siempre a un incumplimiento (default).

---

**C. El "Max Delay" (Máximo Retraso)**
Qué es: El valor más alto de las columnas PAY_0 a PAY_5.

Por qué es vital: Las columnas PAY_n son ordinales. Si un cliente tiene un historial de 0, 0, 0, 2, 0, 0, el modelo podría pensar que el "2" fue un error o ignorarlo por ser un solo dato entre muchos "0". Al crear una columna que diga MAX_DELAY = 2, le estamos diciendo al modelo: "Ojo, este cliente ya tuvo un retraso de 2 meses en el pasado cercano".

---

**D. Tendencia de la Factura (Bill Trend)**
Qué es: Comparar si la deuda de Septiembre es mayor que la de Abril.

Por qué es vital: Captura si el cliente está entrando en una "espiral de deuda".

Tendencia positiva: El cliente gasta más de lo que paga cada mes.

Tendencia negativa: El cliente está saneando sus finanzas.